# Exercise 7: Day-Ahead and Intra-Day Models

**Presenter** <br>
Priyesh Gosai <br>
Energy Systems Modeller <br>
priyesh@innovateimpact.com <br>

**Course Details**<br>
Modelling Integrated Power Markets <br>
Johannesburg, South Africa <br>
20-24 April 2026 <br>


In [46]:
lesson_number = 7
print(f'lesson{lesson_number}')

lesson7


## Prepare Google Colab Environment

In [47]:
#@title Connect to Google Drive {display-mode:"form"}
CONNECT_TO_DRIVE = False #@param {type:"boolean"}

import os
if 'lesson_number' not in locals(): lesson_number = 7
if CONNECT_TO_DRIVE:
    from google.colab import drive
    # Mount Google Drive
    drive.mount('/content/drive')

    # Define the desired working directory path
    working_dir = f'/content/drive/MyDrive/ich-modelling-2026'
    lesson_folder = f'lesson-{lesson_number}'
    lesson_dir = os.path.join(working_dir, lesson_folder)

    # Create the working directory if it doesn't exist
    if not os.path.exists(working_dir):
        os.makedirs(working_dir)
        print(f"Directory '{working_dir}' created.")
    else:
        print(f"Directory '{working_dir}' already exists.")

    # Create the lesson directory if it doesn't exist
    if not os.path.exists(lesson_dir):
        os.makedirs(lesson_dir)
        print(f"Directory '{lesson_dir}' created.")
    else:
        print(f"Directory '{lesson_dir}' already exists.")

    # Change the current working directory to the lesson directory
    os.chdir(lesson_dir)

    print(f"Current working directory: {os.getcwd()}")
else:
    print("Not connecting to Google Drive.")

Not connecting to Google Drive.


In [48]:
#@title Install Packages {display-mode:"form"}
INSTALL_PACKAGES = False #@param {type:"boolean"}

import os

# Check if packages have already been installed in this session to prevent re-installation
if INSTALL_PACKAGES and not os.environ.get('PYPSA_PACKAGES_INSTALLED'):
  !pip install pypsa
  !pip install pypsa[excel] 
  !pip install folium mapclassify cartopy gurobipy
  os.environ['PYPSA_PACKAGES_INSTALLED'] = 'true'
elif not INSTALL_PACKAGES:
  print("Skipping package installation.")
else:
  print("PyPSA packages are already installed for this session.")

Skipping package installation.


In [ ]:
#@title Download the file for this notebook {display-mode:"form"}
DOWNLOAD_FILE = False #@param {type:"boolean"}

import urllib.request
import os

if DOWNLOAD_FILE:
    url = "https://raw.githubusercontent.com/PriyeshGosai/ich_course_2026/46e109dd68c635d83046eec03d2d2a4afc669366/n-01-single-node-v2.xlsx"
    file_name = url.split('/')[-1]  # Extract filename from URL
    filename = file_name
    urllib.request.urlretrieve(url, filename)
    print(f"File downloaded successfully: {os.path.abspath(filename)}")

else:
    print("Skipping file download.")

Skipping file download.


## Functions

In [50]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_supply_demand_curve(supplier_volume, supplier_prices, consumer_volume, consumer_prices, bus_name, snapshot_idx, n):
    """
    Plot supply and demand curves for a given bus and snapshot.
    
    Parameters:
    - supplier_volume: DataFrame of supplier volumes
    - supplier_prices: DataFrame of supplier prices
    - consumer_volume: DataFrame of consumer volumes
    - consumer_prices: DataFrame of consumer prices
    - bus_name: Name of the bus to plot
    - snapshot_idx: Index of the snapshot to plot
    - n: PyPSA network object
    """
    
    # Supply curve
    supply_at_bus = supplier_volume.iloc[snapshot_idx][supplier_volume.iloc[snapshot_idx] > 0]
    supply_prices_at_bus = supplier_prices.loc[n.snapshots[snapshot_idx], supply_at_bus.index]
    
    supply_curve = pd.DataFrame({
        'dispatch': supply_at_bus.values,
        'price': supply_prices_at_bus.values
    }, index=supply_at_bus.index)
    
    supply_curve = supply_curve[supply_curve['dispatch'] > 0].sort_values('price')
    
    # Demand curve
    demand_at_bus = consumer_volume.iloc[snapshot_idx][consumer_volume.iloc[snapshot_idx] > 0]
    demand_prices_at_bus = consumer_prices.loc[n.snapshots[snapshot_idx], demand_at_bus.index]
    
    demand_curve = pd.DataFrame({
        'dispatch': demand_at_bus.values,
        'price': demand_prices_at_bus.values
    }, index=demand_at_bus.index)
    
    demand_curve = demand_curve[demand_curve['dispatch'] > 0].sort_values('price', ascending=False)
    
    # Calculate max for xlim
    max_dispatch = max(supply_curve['dispatch'].sum(), demand_curve['dispatch'].sum())
    
    # Plot
    plt.figure(figsize=(10, 6))
    
    plt.step(
        np.cumsum([0] + supply_curve['dispatch'].tolist()),
        supply_curve['price'].iloc[:1].tolist() + supply_curve['price'].tolist(),
        where='pre',
        linewidth=2,
        color='steelblue',
        label='Supply'
    )
    
    if len(demand_curve) > 0:
        plt.step(
            np.cumsum([0] + demand_curve['dispatch'].tolist()),
            demand_curve['price'].iloc[:1].tolist() + demand_curve['price'].tolist(),
            where='pre',
            linewidth=2,
            color='coral',
            label='Demand'
        )
    
    plt.xlim(0, max_dispatch)
    plt.xlabel('Cumulative Dispatch (MW)')
    plt.ylabel('Price (ZAR/MWh)')
    plt.title(f'Supply-Demand Curve - {bus_name} at {n.snapshots[snapshot_idx]}(Snapshot {snapshot_idx})')
    plt.legend()
    plt.tight_layout()
    plt.show()




In [51]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pandas as pd
from datetime import datetime

# ============================================================================
# MATPLOTLIB VERSION
# ============================================================================

def plot_dispatch_matplotlib(n, start_date=None, end_date=None, aggregate_data=True):
    """
    Plot dispatch with Matplotlib.
    
    Parameters:
    -----------
    n : pypsa.Network
        The PyPSA network object containing the data to plot.
    start_date : str, optional
        Start date in format 'YYYY-MM-DD' (always 00:00)
    end_date : str, optional
        End date in format 'YYYY-MM-DD' (always 23:00)
    aggregate_data : bool, default True
        If True, aggregate all generators/storage units.
        If False, show individual plant contributions.
    """
    
    # 1. Get generator names (excluding elastic demand)
    non_elastic_generators = n.generators.static[n.generators.static.sign >= 0].index
    
    # 2. Split dispatch and storage
    gen_dispatch = n.generators.dynamic.p[non_elastic_generators]  # Exclude elastic demand
    storage_discharge = n.storage_units.dynamic.p[n.storage_units.dynamic.p >= 0]
    storage_charge = n.storage_units.dynamic.p[n.storage_units.dynamic.p < 0].abs()

    # 3. Prepare data - convert charging to negative
    charging = -storage_charge.fillna(0)  # Negative for below x-axis
    generators = gen_dispatch.fillna(0)
    discharging = storage_discharge.fillna(0)

    # 4. Get load data
    fixed_load = n.loads.dynamic.p.sum(axis=1)
    elastic_generators = n.generators.static[n.generators.static.sign < 0].index
    elastic_load = n.generators.dynamic.p[elastic_generators].sum(axis=1)

    # Filter data by date range
    if start_date and end_date:
        start_dt = pd.Timestamp(start_date + ' 00:00')
        end_dt = pd.Timestamp(end_date + ' 23:00')
        
        mask = (charging.index >= start_dt) & (charging.index <= end_dt)
        charging_filtered = charging[mask]
        generators_filtered = generators[mask]
        discharging_filtered = discharging[mask]
        fixed_load_filtered = fixed_load[mask]
        elastic_load_filtered = elastic_load[mask]
    else:
        charging_filtered = charging
        generators_filtered = generators
        discharging_filtered = discharging
        fixed_load_filtered = fixed_load
        elastic_load_filtered = elastic_load
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Stacked area plot
    if aggregate_data:
        # Aggregated view
        ax.stackplot(
            charging_filtered.index,
            charging_filtered.sum(axis=1),  # Negative
            generators_filtered.sum(axis=1),
            discharging_filtered.sum(axis=1),
            labels=['Charging Storage Units', 'Generator Dispatch', 'Discharging Storage Units'],
            alpha=0.7,
            colors=['#1f77b4', '#ff7f0e', '#2ca02c']
        )
    else:
        # Individual plant view
        # Charging storage units (negative)
        charging_sum = charging_filtered.sum(axis=1)
        ax.fill_between(charging_filtered.index, 0, charging_sum, alpha=0.7, color='#1f77b4', label='Charging Storage Units')
        
        # Generator dispatch (individual)
        ax.stackplot(
            generators_filtered.index,
            *[generators_filtered[col] for col in generators_filtered.columns],
            alpha=0.7,
            labels=generators_filtered.columns
        )
        
        # Discharging storage units (individual, stacked above generators)
        gen_cumsum = generators_filtered.sum(axis=1)
        for col in discharging_filtered.columns:
            ax.fill_between(discharging_filtered.index, gen_cumsum, gen_cumsum + discharging_filtered[col], 
                           alpha=0.7, label=col)
            gen_cumsum = gen_cumsum + discharging_filtered[col]
    
    # Overlay loads
    load_total = fixed_load_filtered + elastic_load_filtered
    ax.plot(fixed_load_filtered.index, fixed_load_filtered, color='black', linewidth=2.5, label='Total Fixed Load', zorder=5)
    ax.plot(load_total.index, load_total, color='red', linewidth=2.5, linestyle='--', label='Total Load (Fixed + Elastic)', zorder=5)
    
    # Formatting
    ax.axhline(y=0, color='black', linewidth=0.8, zorder=4)
    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Power (MW)', fontsize=12)
    ax.set_title('Dispatch with Storage Charging/Discharging and Load', fontsize=14)
    ax.set_xlim(charging_filtered.index.min(), charging_filtered.index.max())
    ax.legend(loc='upper left', fontsize=9, bbox_to_anchor=(1.01, 1))
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# ============================================================================
# PLOTLY VERSION
# ============================================================================

def plot_dispatch_plotly(n, aggregate_data=True):
    """
    Plot dispatch with Plotly (interactive).
    
    Parameters:
    -----------
    n : pypsa.Network
        The PyPSA network object containing the data to plot.
    aggregate_data : bool, default True
        If True, aggregate all generators/storage units.
        If False, show individual plant contributions.
    """
    
    # 1. Get generator names (excluding elastic demand)
    non_elastic_generators = n.generators.static[n.generators.static.sign >= 0].index
    
    # 2. Split dispatch and storage
    gen_dispatch = n.generators.dynamic.p[non_elastic_generators]  # Exclude elastic demand
    storage_discharge = n.storage_units.dynamic.p[n.storage_units.dynamic.p >= 0]
    storage_charge = n.storage_units.dynamic.p[n.storage_units.dynamic.p < 0].abs()

    # 3. Prepare data - convert charging to negative
    charging = -storage_charge.fillna(0)  # Negative for below x-axis
    generators = gen_dispatch.fillna(0)
    discharging = storage_discharge.fillna(0)

    # 4. Get load data
    fixed_load = n.loads.dynamic.p.sum(axis=1)
    elastic_generators = n.generators.static[n.generators.static.sign < 0].index
    elastic_load = n.generators.dynamic.p[elastic_generators].sum(axis=1)

    fig = go.Figure()
    
    if aggregate_data:
        # Aggregated view
        fig.add_trace(go.Scatter(
            x=charging.index,
            y=charging.sum(axis=1),
            mode='lines',
            name='Charging Storage Units',
            stackgroup='one',
            fillcolor='rgba(31, 119, 180, 0.7)',
            line=dict(color='rgba(31, 119, 180, 0.7)')
        ))
        
        fig.add_trace(go.Scatter(
            x=generators.index,
            y=generators.sum(axis=1),
            mode='lines',
            name='Generator Dispatch',
            stackgroup='one',
            fillcolor='rgba(255, 127, 14, 0.7)',
            line=dict(color='rgba(255, 127, 14, 0.7)')
        ))
        
        fig.add_trace(go.Scatter(
            x=discharging.index,
            y=discharging.sum(axis=1),
            mode='lines',
            name='Discharging Storage Units',
            stackgroup='one',
            fillcolor='rgba(44, 160, 44, 0.7)',
            line=dict(color='rgba(44, 160, 44, 0.7)')
        ))
    else:
        # Individual plant view
        # Charging storage units (negative, non-stacking)
        for col in charging.columns:
            fig.add_trace(go.Scatter(
                x=charging.index,
                y=charging[col],
                mode='lines',
                name=f'{col} (Charging)',
                stackgroup='charging',
                fillcolor='rgba(31, 119, 180, 0.7)',
                line=dict(color='rgba(31, 119, 180, 0.7)')
            ))
        
        # Generator dispatch (individual, stacked positive)
        for col in generators.columns:
            fig.add_trace(go.Scatter(
                x=generators.index,
                y=generators[col],
                mode='lines',
                name=col,
                stackgroup='generation',
                fillcolor=None,
                line=dict(width=1)
            ))
        
        # Discharging storage units (individual, stacked positive)
        for col in discharging.columns:
            fig.add_trace(go.Scatter(
                x=discharging.index,
                y=discharging[col],
                mode='lines',
                name=f'{col} (Discharging)',
                stackgroup='generation',
                fillcolor=None,
                line=dict(width=1)
            ))
    
    # Overlay loads
    fig.add_trace(go.Scatter(
        x=fixed_load.index,
        y=fixed_load,
        mode='lines',
        name='Total Fixed Load',
        line=dict(color='black', width=2.5),
        yaxis='y'
    ))
    
    load_total = fixed_load + elastic_load
    fig.add_trace(go.Scatter(
        x=load_total.index,
        y=load_total,
        mode='lines',
        name='Total Load (Fixed + Elastic)',
        line=dict(color='red', width=2.5, dash='dash'),
        yaxis='y'
    ))
    
    # Layout
    fig.update_layout(
        title='Dispatch with Storage Charging/Discharging and Load',
        xaxis_title='Time',
        yaxis_title='Power (MW)',
        hovermode='x unified',
        height=600,
        template='plotly_white'
    )
    
    fig.show()

#### Case Description

Simulate least cost dispatch given fixed capacity availability with unit commitment and rolling horizon optimization.

##### PyPSA Features Used

| Feature | Method |
|---------|--------|
| Inspect dispatch data | Programmatically |
| Activate unit commitment for dispatchable generators | Programmatically |
| Apply quadratic marginal costs and standby costs | Programmatically |
| Run rolling horizon optimisation | Programmatically |
| Calculate marginal prices | Programmatically |
| Plot supply and demand | Programmatically |

##### Non-Standard PyPSA Features

These features are not standard in PyPSA but can be implemented through component manipulation:


#### Lesson Tasks

1. Inspect market model results
2. Compare results with and without unit commitment
3. Compare how rolling horizon optimisation differs from full period optimisation

## Model

### User inputs

In [ ]:
week = 10 # False : Insert week number (1-52) or False for full year simulation

### Import packages, read input data and optimize

In [53]:
# Import packages
import pypsa                        # pypsa toolbox
import pandas as pd                 # pandas toolbox for handling tables
import plotly.graph_objects as go   # interactive plotting toolbox
import numpy as np

# Supress unnecessary warnings
pd.set_option('future.no_silent_downcasting', True)

# The default plotting tool for pandas is matplotlib which only provies static plots. 
# Changing to plotly allows for interactive plotting. 
pd.set_option('plotting.backend', 'plotly')

# When pypsa version 1 was launched, a new set of commands were created. 
# This option ensures that we use the new commands. 
pypsa.options.api.new_components_api = True

In [54]:
import pypsa
import pandas as pd
import numpy as np

print(f'Load Network from {file_name}')
n = pypsa.Network(file_name)


n.meta = pd.read_excel(file_name, sheet_name='meta', index_col=0, header=None)[1].to_dict()

# Disable elastic demand
n.generators.static.loc[n.generators.static.carrier == 'demand_elastic', 'active'] = False


if week is not False: n.set_snapshots(n.snapshots[n.snapshots.isocalendar().week == week])

# Create day-ahead and intra-day networks
n_DA = n.copy()
n_ID = n.copy()


# Day Ahead Market

# Set committable to True for dispatchable generators
dispatchable_carriers = ['coal', 'gas']
n_DA.generators.static.loc[n_DA.generators.static.carrier.isin(dispatchable_carriers), 'committable'] = True

# Run DA optimization
n_DA.optimize(include_objective_constant=True)






Load Network from n_02_single_node_v2.xlsx


INFO:pypsa.network.io:Imported network 'ich-single-node-model' has buses, carriers, generators, global_constraints, lines, line_types, links, loads, processes, shapes, shunt_impedances, storage_units, stores, sub_networks, transformers, transformer_types
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.11s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 5880 primals, 13449 duals
Objective: 1.03e+08
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-status-p-fixed-upper, Generator-start_up-p-fixed-upper, Generator-shut_down-p-fixed-upper, Generator-fix-p-lower, Generator-fix-p-upper, Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Generator-com-up-time, Generator-com-down-time, Generator-com-status-min_up_time_must_stay_up, Generator-p-ramp_limit_up, Gener

('ok', 'optimal')

In [55]:
# Calculate marginal prices for DA markets
supply_gens = n_DA.generators.static[n_DA.generators.static.type == 'Supply'].index.tolist()
supply_storage = n_DA.storage_units.static.index.tolist()

# Supplier volume (generation)
supplier_volume = pd.concat([
    n_DA.generators.dynamic.p[supply_gens],
    n_DA.storage_units.dynamic.p[n_DA.storage_units.dynamic.p >= 0].fillna(0)
], axis=1)

# Demand volume (loads and demand generators)
demand_gens = n_DA.generators.static[n_DA.generators.static.type == 'Demand'].index.tolist()
demand_loads = n_DA.loads.static[n_DA.loads.static.type == 'Demand'].index.tolist()

consumer_volume = pd.concat([
    n_DA.loads.dynamic.p[demand_loads],
    n_DA.generators.dynamic.p[demand_gens],
    n_DA.storage_units.dynamic.p[n_DA.storage_units.dynamic.p < 0].fillna(0)
], axis=1).abs()

# Extract supplier marginal costs
supplier_prices = pd.DataFrame(index=n_DA.snapshots)

for gen in supply_gens:
    if gen in n_DA.generators.dynamic.marginal_cost.columns:
        supplier_prices[gen] = n_DA.generators.dynamic.marginal_cost[gen]
    else:
        supplier_prices[gen] = n_DA.generators.static.loc[gen, 'marginal_cost']

for storage in supply_storage:
    if storage in n_DA.storage_units.dynamic.marginal_cost.columns:
        supplier_prices[storage] = n_DA.storage_units.dynamic.marginal_cost[storage]
    else:
        supplier_prices[storage] = n_DA.storage_units.static.loc[storage, 'marginal_cost']

# DA marginal price = max marginal cost of suppliers actually supplying
da_marginal_prices = supplier_prices[supplier_volume > 0].max(axis=1)

In [56]:
# Intra-Day Market
# Get dispatchable generator names
dispatchable_gens = n_DA.generators.static.index[n_DA.generators.static.carrier.isin(dispatchable_carriers)]
status_DA = n_DA.generators.dynamic.status[dispatchable_gens]

# Update n_ID p_max_pu using DA results
# Broadcast static values across all time steps and multiply by status
n_ID.generators.dynamic.p_max_pu[dispatchable_gens] = (
    n_ID.generators.static.p_max_pu[dispatchable_gens].values * status_DA.values
)

# Update n_ID p_min_pu using DA results
n_ID.generators.dynamic.p_min_pu[dispatchable_gens] = (
    n_ID.generators.static.p_min_pu[dispatchable_gens].values * status_DA.values
)

# Introduce uncertainty: reduce p_max_pu with random factors (0.85-1.15, clipped to 0-1)
random_factors = np.random.uniform(0.85, 1.15, size=n_ID.generators.dynamic.p_max_pu.shape).clip(0, 1)
n_ID.generators.dynamic.p_max_pu = n_ID.generators.dynamic.p_max_pu * random_factors



# Reduce to first 24 hours
n_ID.set_snapshots(n_ID.snapshots[:24])

# Run intra-day optimization with rolling horizon
n_ID.optimize.optimize_with_rolling_horizon(horizon=6, overlap=0)


INFO:pypsa.optimization.abstract:Optimizing network for snapshot horizon [2026-03-02 00:00:00:2026-03-02 05:00:00] (1/4).
c:\Users\PriyeshGosai\anaconda3\envs\pypsa-model-env\Lib\site-packages\pypsa\optimization\abstract.py:545: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  status, condition = n.optimize(sns, **kwargs)
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.07s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 120 primals, 288 duals
Objective: 7.19e+06
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-p-ramp_limit_up, Generator-p-ram

PyPSA Network 'ich-single-node-model'
-------------------------------------
Components:
 - Bus: 1
 - Carrier: 12
 - Generator: 19
 - Load: 3
 - StorageUnit: 2
 - SubNetwork: 1
Snapshots: 24

In [63]:
# Calculate revenues from DA prices and ID dispatch

# Get supply generators and storage
supply_gens = n_DA.generators.static[n_DA.generators.static.type == 'Supply'].index.tolist()
supply_storage = n_DA.storage_units.static.index.tolist()

# Extract ID dispatch (first 24 hours match DA pricing)
id_dispatch_gens = n_ID.generators.dynamic.p[supply_gens]
id_dispatch_storage = n_ID.storage_units.dynamic.p[supply_storage]

# Align DA marginal prices to ID snapshots (first 24 hours)
da_prices_for_id = da_marginal_prices[:24]

# Calculate revenues for each generator
revenues_gens = pd.DataFrame(index=id_dispatch_gens.index)

for gen in supply_gens:
    if gen in id_dispatch_gens.columns:
        # Revenue = dispatch * DA marginal price
        revenues_gens[gen] = id_dispatch_gens[gen] * da_prices_for_id.values

# Calculate revenues for storage units (only when discharging, p > 0)
revenues_storage = pd.DataFrame(index=id_dispatch_storage.index)

for storage in supply_storage:
    if storage in id_dispatch_storage.columns:
        # Only count positive dispatch (discharging)
        revenues_storage[storage] = id_dispatch_storage[storage].clip(lower=0) * da_prices_for_id.values

# Total revenues by timestamp
total_revenue_by_time = pd.concat([revenues_gens, revenues_storage], axis=1).sum(axis=1)

# Total revenues by generator/storage unit
total_revenue_by_unit = pd.concat([revenues_gens, revenues_storage], axis=1).sum(axis=0)

print("\nTotal Revenue by Unit (Million ZAR):")
revenue_df = pd.DataFrame((total_revenue_by_unit / 1e6).round(3), columns=['Revenue (M ZAR)'])
revenue_df




Total Revenue by Unit (Million ZAR):


,Revenue (M ZAR)
Athlone Coal 1,11.626
Ankerlig Gas,10.990
Salt River Coal 1,9.717
Salt River Coal 2,7.999
Solar 1,12.322
Loadshedding WC,0.000
Future OCGT,7.785
Wind 1,3.251
Wind 2,4.031
Wind 3,8.427


### Dispatch results

In [58]:
plot_dispatch_plotly(n_DA,aggregate_data=False)

In [59]:
plot_dispatch_plotly(n_ID,aggregate_data=False)